# 00 — WRDS Setup & Connection Test

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 14866196  
**Purpose:** Verify that the WRDS Python connection works and that the expected databases are accessible under the UvA subscription.  

---

## What this notebook does

1. Installs and imports required packages
2. Connects to WRDS using credentials from `.env`
3. Lists all libraries (databases) available to your account
4. Checks specifically for the databases needed for this thesis
5. Saves a connection log to `logs/`

**This notebook does NOT pull any data. It only tests access.**

---

> ⚠️ **Before running:** make sure you have:
> 1. Copied `.env.example` → `.env` and filled in your WRDS username
> 2. Installed dependencies: `pip install -r requirements.txt`

## Step 1 — Imports

In [3]:
import os
import sys
import datetime
import wrds
import pandas as pd
from dotenv import load_dotenv

# Add src/ to path so we can import wrds_utils
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds

print(f"Python     : {sys.version}")
print(f"wrds pkg   : installed")
print(f"pandas     : {pd.__version__}")
print(f"Run time   : {datetime.datetime.now()}")

Python     : 3.13.3 (v3.13.3:6280bb54784, Apr  8 2025, 10:47:54) [Clang 15.0.0 (clang-1500.3.9.4)]
wrds pkg   : installed
pandas     : 2.2.3
Run time   : 2026-03-07 18:28:42.757861


## Step 2 — Connect to WRDS

The first time you run this, WRDS will ask you to enter your password interactively and will offer to save a `pgpass` file so you don't have to re-enter it every time. Accept this — it is safe and standard.

In [4]:
# Load .env credentials
load_dotenv()

# Connect
conn = connect_wrds()

Connecting to WRDS as: basar


Enter your WRDS username [basar]: 
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connection established.


## Step 3 — List all accessible libraries

This shows every database (called a 'library' in WRDS terminology) that your account can query.

In [5]:
# List all libraries available to your account
libraries = conn.list_libraries()

print(f"Total libraries accessible: {len(libraries)}")
print("\nFull list:")
for lib in sorted(libraries):
    print(f"  {lib}")

Total libraries accessible: 252

Full list:
  aha_sample
  ahasamp
  audit
  audit_acct_os
  audit_audit_comp
  audit_common
  audit_europe
  auditsmp
  auditsmp_all
  bank
  bank_all
  bank_premium_samp
  banksamp
  block
  block_all
  boardex_trial
  boardsmp
  bvd
  bvd_amadeus_trial
  bvd_bvdbankf
  bvd_bvdbankf_trial
  bvd_orbis_large
  bvd_orbis_medium
  bvd_orbis_small
  bvd_orbis_trial
  bvdsamp
  calcbench_trial
  calcbnch
  candid_samp
  cboe
  cboe_all
  cboe_sample
  cboesamp
  cddsamp
  ciq
  ciq_common
  ciqsamp
  ciqsamp_capstrct
  ciqsamp_common
  ciqsamp_keydev
  ciqsamp_pplintel
  ciqsamp_ratings
  ciqsamp_transactions
  ciqsamp_transcripts
  cisdmsmp
  columnar
  comp
  comp_bank_daily
  comp_execucomp
  comp_global_daily
  comp_na_daily_all
  comp_segments_hist_daily
  compsamp
  compsamp_all
  compsamp_snapshot
  compseg
  contrib
  contrib_as_filed_financials
  contrib_bond_firm_link
  contrib_ceo_turnover
  contrib_char_returns
  contrib_corporate_culture
  contr

## Step 4 — Check thesis-critical databases

We check for each database we need. Green = accessible. Red = not found.

In [7]:
# Define the databases we need to check
# Format: (wrds_library_name, description, priority)
DATABASES_TO_CHECK = [
    # --- CORE (non-negotiable) ---
    ("optionm",         "OptionMetrics IvyDB US (full)",          "CORE"),
    ("optionmeurope",   "OptionMetrics IvyDB Europe (full)",       "CORE"),
    ("optionmsamp_us",     "OptionMetrics IvyDB US (sample)",         "CORE - sample"),
    ("optionmsamp_europe",  "OptionMetrics IvyDB Europe (sample)",     "CORE - sample"),
    ("cboe",            "CBOE VIX Daily",                          "CORE"),
    ("frb",             "Federal Reserve Board Rates",             "CORE"),
    ("crsp",            "CRSP Stock & Indexes",                    "CORE"),
    ("comp",            "Compustat Global Daily",                  "CORE"),
    # --- CONTROLS ---
    ("ff",              "Fama-French Factors",                     "CONTROL"),
    ("djones",          "Dow Jones Averages",                      "CONTROL"),
    ("pwt",             "Penn World Tables",                       "CONTROL"),
    ("macrofin",        "Macro Finance Society",                   "CONTROL"),
    ("markit",          "Markit CDS / CDX",                        "CONTROL"),
    ("phlx",            "PHLX Currency Options & IV",              "CONTROL"),
    # --- SUPPLEMENTARY ---
    ("wrdsapps",        "WRDS Applications (indices, linking)",    "SUPPLEMENTARY"),
    ("tr",              "Thomson Reuters / Datastream samples",    "SUPPLEMENTARY"),
]

print(f"{'Database':<20} {'Description':<45} {'Priority':<15} {'Access'}")
print("-" * 100)

results = []
for lib, desc, priority in DATABASES_TO_CHECK:
    accessible = lib in libraries
    status = "✓ ACCESSIBLE" if accessible else "✗ NOT FOUND"
    print(f"{lib:<20} {desc:<45} {priority:<15} {status}")
    results.append({"library": lib, "description": desc,
                    "priority": priority, "accessible": accessible})

results_df = pd.DataFrame(results)

Database             Description                                   Priority        Access
----------------------------------------------------------------------------------------------------
optionm              OptionMetrics IvyDB US (full)                 CORE            ✗ NOT FOUND
optionmeurope        OptionMetrics IvyDB Europe (full)             CORE            ✗ NOT FOUND
optionmsamp_us       OptionMetrics IvyDB US (sample)               CORE - sample   ✓ ACCESSIBLE
optionmsamp_europe   OptionMetrics IvyDB Europe (sample)           CORE - sample   ✓ ACCESSIBLE
cboe                 CBOE VIX Daily                                CORE            ✓ ACCESSIBLE
frb                  Federal Reserve Board Rates                   CORE            ✓ ACCESSIBLE
crsp                 CRSP Stock & Indexes                          CORE            ✓ ACCESSIBLE
comp                 Compustat Global Daily                        CORE            ✓ ACCESSIBLE
ff                   Fama-French Factors   

## Step 5 — For accessible databases, list their tables

For each database we can access, we list the tables inside it. This tells us what data is actually available to query.

In [8]:
# List tables for each accessible thesis-relevant database
accessible_libs = [r["library"] for r in results if r["accessible"]]

table_registry = {}

for lib in accessible_libs:
    try:
        tables = conn.list_tables(library=lib)
        table_registry[lib] = tables
        print(f"\n--- {lib} ({len(tables)} tables) ---")
        for t in sorted(tables):
            print(f"  {t}")
    except Exception as e:
        print(f"\n--- {lib} --- ERROR: {e}")
        table_registry[lib] = []


--- optionmsamp_us (12 tables) ---
  distrd
  exchgd
  hvold2014
  opinfd
  opprcd2014
  opvold
  secnmd
  secprd
  secprd2014
  securd1
  stdopd2014
  vsurfd2014

--- optionmsamp_europe (12 tables) ---
  distribution
  historical_volatility
  option_history
  option_price_2013
  optionmeurnames
  security_name
  security_price
  std_option_price_2013
  tick_option_price_2013
  tick_std_option_price_2013
  tick_volatility_surface_2013
  volatility_surface_2013

--- cboe (90 tables) ---
  cboe
  deltanumber
  eqfactor
  eqhvol
  eqhvolmap
  eqmaster
  eqprice
  eqsplitdiv
  eqsplitdivsupp
  eqsplitdivtype
  ivborrowrate
  ivcmdr
  ivcmpr
  ivdivmap
  ivexdr
  ivexpr
  ivlisted
  ivlisted_1998
  ivlisted_1999
  ivlisted_2000
  ivlisted_2001
  ivlisted_2002
  ivlisted_2003
  ivlisted_2004
  ivlisted_2005
  ivlisted_2006
  ivlisted_2007
  ivlisted_2008
  ivlisted_2009
  ivlisted_2010
  ivlisted_2011
  ivlisted_2012
  ivlisted_2013
  ivlisted_2014
  ivlisted_2015
  ivlisted_2016
  ivlisted

## Step 6 — Save connection log

In [9]:
import json

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"../logs/wrds_access_check_{timestamp}.json"

log = {
    "timestamp": timestamp,
    "wrds_username": os.getenv("WRDS_USERNAME"),
    "total_libraries_accessible": len(libraries),
    "thesis_database_check": results,
    "table_registry": {k: v for k, v in table_registry.items()}
}

with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f"Log saved to: {log_path}")

Log saved to: ../logs/wrds_access_check_20260307_183542.json


## Step 7 — Close connection

In [10]:
conn.close()
print("WRDS connection closed.")
print("\n✓ Setup test complete. Proceed to notebook 01_data_pull_inspect.ipynb")

WRDS connection closed.

✓ Setup test complete. Proceed to notebook 01_data_pull_inspect.ipynb
